In [4]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-e6b94c71-1db7-5ad2-a95a-9de8bd50bb1a"
# ----------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import time
from tqdm.auto import tqdm


LEARNING_RATE = 0.0005

BATCH_SIZE = 32
NUM_EPOCHS = 30 
NUM_CLASSES = 12

TRAIN_DATA_FILE = 'train.h5'
TEST_DATA_FILE = 'test.h5'
TEST_LABELS_FILE = 'test_labels.csv' 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def load_and_preprocess_data():
    """Loads, concatenates, transposes, and scales all data."""
    
    print("Loading labels...")
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        y_train = np.array(hdf.get('Class/1'))

    y_test = pd.read_csv(TEST_LABELS_FILE, header=None).values.squeeze()

    print("Loading and concatenating training spectra...")
    all_train_spectra = []
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        train_keys = [f"{i:03d}" for i in range(1, 101)]
        for key in tqdm(train_keys, desc="Training data"):
            data_block = np.array(hdf[f'Spectra/{key}'])
            all_train_spectra.append(data_block.T)
    
    X_train_raw = np.concatenate(all_train_spectra, axis=0)

    print("Loading and concatenating test spectra...")
    all_test_spectra = []
    with h5py.File(TEST_DATA_FILE, 'r') as hdf:
        test_keys = ['1', '2']
        for key in tqdm(test_keys, desc="Test data    "):
            data_block = np.array(hdf[f'UNKNOWN/{key}'])
            all_test_spectra.append(data_block.T)
            
    X_test_raw = np.concatenate(all_test_spectra, axis=0)

    print(f"Raw X_train shape: {X_train_raw.shape}, Raw y_train shape: {y_train.shape}")
    print(f"Raw X_test shape: {X_test_raw.shape}, Raw y_test shape: {y_test.shape}")

    print("Scaling data...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    print("Data loading and preprocessing complete.")
    return X_train_scaled, y_train, X_test_scaled, y_test


class LibsDataset(Dataset):
    """Custom PyTorch Dataset for LIBS spectra (for CNN input)."""
    def __init__(self, data, labels):
        # CNN expects: (N, Channels, Length)
        self.X = torch.tensor(data, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(labels, dtype=torch.long)
        
        # Labels are 1-indexed (1-12), convert to 0-indexed (0-11)
        self.y = self.y - 1 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]



class CNN_LSTM_V3(nn.Module):
    """
    V3 Hybrid Model: Deeper CNN frontend (4 blocks)
    Sequence: 40002 -> 10000 -> 2500 -> 250 -> 50
    """
    def __init__(self, num_classes=12, lstm_hidden_size=64):
        super(CNN_LSTM_V3, self).__init__()
        
        # --- CNN Feature Extractor ---
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4), # 40002 -> 10000
            nn.Dropout(0.3)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4), # 10000 -> 2500
            nn.Dropout(0.3)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=10, stride=10), # 2500 -> 250
            nn.Dropout(0.4)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=5, stride=5), # 250 -> 50
            nn.Dropout(0.4)
        )
        
        # --- LSTM Sequence Processor ---
        # Input to LSTM: (Batch, SeqLength=50, Features=256)
        self.lstm = nn.LSTM(
            input_size=256,          
            hidden_size=lstm_hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # --- Classifier ---
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_size * 2, 128), # 64 * 2
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # 1. Pass through all 4 CNN blocks
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        
        # 2. Prepare for LSTM: (Batch, 256, 50) -> (Batch, 50, 256)
        x = x.permute(0, 2, 1)
        
        # 3. Pass through LSTM
        all_outputs, (h_n, c_n) = self.lstm(x)
        
        # 4. Concatenate final forward & backward hidden states
        x = h_n.permute(1, 0, 2).reshape(-1, 128) # 64 * 2
        
        # 5. Classify
        x = self.classifier(x)
        return x



def train_model(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, (spectra, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        spectra = spectra.to(device)
        labels = labels.to(device)
        
        outputs = model(spectra)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
            
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'--- Epoch {epoch+1} Summary ---')
    print(f'Average Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s')



def evaluate_model(model, test_loader, criterion):
    print("\nEvaluating model on test data...")
    model.eval()
    
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for spectra, labels in tqdm(test_loader, desc="Evaluating"):
            spectra = spectra.to(device)
            labels = labels.to(device)
            
            outputs = model(spectra)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f'Test Loss: {avg_loss:.4f}')
    print(f'Test Accuracy: {accuracy * 100:.2f}%')
    
    print("\n--- Classification Report ---")
    target_names = [f"Class {i+1}" for i in range(NUM_CLASSES)]
    print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=target_names, digits=4, zero_division=0))



if __name__ == "__main__":
    
    print(f"Using device: {device}")
    
    # --- Data ---
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    print("\nCalculating class weights for weighted loss...")
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print(f"Weights calculated: {class_weights_tensor}")

    print("\nCreating PyTorch Datasets and DataLoaders...")
    train_dataset = LibsDataset(X_train, y_train)
    test_dataset = LibsDataset(X_test, y_test)
    

    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print(f"Total training samples: {len(train_dataset)}")
    print(f"Total test samples: {len(test_dataset)}")

    # --- Model ---
    model = CNN_LSTM_V3(num_classes=NUM_CLASSES).to(device)
    print("\nNEW CNN-LSTM (V3) Model architecture created and moved to GPU.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
    print("Using Weighted Loss, Adam Optimizer, and StepLR Scheduler.")

    # --- Training ---
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    total_start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        train_model(model, train_loader, criterion, optimizer, epoch)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"--- End of Epoch {epoch+1}, Current LR: {current_lr} ---")
        
    total_end_time = time.time()
    print(f"\n--- Training Finished ---")
    print(f"Total training time: {(total_end_time - total_start_time)/60:.2f} minutes")

    # --- Evaluation ---
    evaluate_model(model, test_loader, criterion)

Using device: cuda
Loading labels...
Loading and concatenating training spectra...


Training data:   0%|          | 0/100 [00:00<?, ?it/s]

Loading and concatenating test spectra...


Test data    :   0%|          | 0/2 [00:00<?, ?it/s]

Raw X_train shape: (50000, 40002), Raw y_train shape: (50000,)
Raw X_test shape: (20000, 40002), Raw y_test shape: (20000,)
Scaling data...
Data loading and preprocessing complete.

Calculating class weights for weighted loss...
Weights calculated: tensor([0.9259, 0.5556, 1.6667, 1.3889, 1.1905, 0.9259, 1.6667, 1.3889, 0.5556,
        1.6667, 0.6944, 1.3889], device='cuda:0')

Creating PyTorch Datasets and DataLoaders...
Total training samples: 50000
Total test samples: 20000

NEW CNN-LSTM (V3) Model architecture created and moved to GPU.
Total parameters: 316,748
Using Weighted Loss, Adam Optimizer, and StepLR Scheduler.

Starting training for 30 epochs...


...

--- Epoch 29 Summary ---
Average Training Loss: 0.2146, Time: 63.63s
--- End of Epoch 29, Current LR: 5e-05 ---


Epoch 30/30:   0%|          | 0/1563 [00:00<?, ?it/s]

--- Epoch 30 Summary ---
Average Training Loss: 0.2085, Time: 64.41s
--- End of Epoch 30, Current LR: 5e-06 ---

--- Training Finished ---
Total training time: 31.82 minutes

Evaluating model on test data...


Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]

Test Loss: 2.8180
Test Accuracy: 55.52%

--- Classification Report ---
              precision    recall  f1-score   support

     Class 1     0.4544    0.4355    0.4447      3146
     Class 2     0.6506    0.5641    0.6042      3129
     Class 3     0.0391    0.0203    0.0267       543
     Class 4     0.5962    0.7966    0.6820      1603
     Class 5     0.3030    0.7137    0.4254      1048
     Class 6     0.6500    0.7118    0.6795      1589
     Class 7     0.0000    0.0000    0.0000       560
     Class 8     0.6118    0.8195    0.7006      1546
     Class 9     0.7439    0.5424    0.6273      3127
    Class 10     0.2798    0.9805    0.4354       514
    Class 11     0.9759    0.4175    0.5848      3195
    Class 12     0.0000    0.0000    0.0000         0

    accuracy                         0.5552     20000
   macro avg     0.4420    0.5002    0.4342     20000
weighted avg     0.6163    0.5552    0.5530     20000



In [3]:
import torch
print(f"Is CUDA available in this notebook? {torch.cuda.is_available()}")

Is CUDA available in this notebook? True
